# LLMChain

LLMChain 是 LangChain 中最简单的链，作为其他复杂 Chains 和 Agents 的内部调用，被广泛应用。

一个LLMChain由PromptTemplate和语言模型（LLM or Chat Model）组成。它使用直接传入（或 memory 提供）的 key-value 来规范化生成 Prompt Template（提示模板），并将生成的 prompt （格式化后的字符串）传递给大模型，并返回大模型输出。

```
Chain（抽象基类）
     ↓
LLMChain（最常用、最基础的具体链）
```

LLMChain 直接继承 Chain 基类，复用基类所有能力：
memory / callbacks / verbose / tags / metadata / invoke()


PromptTemplate → LLM/ChatModel → OutputParser 是所有复杂 Chain（顺序链、路由链）的最小组成单元。

## LLMChain 核心内置属性

prompt: BasePromptTemplate   # 提示词模板
llm: BaseLanguageModel       # 大模型 / 聊天模型
output_parser: Optional[BaseOutputParser] = None  # 可选输出解析器

In [4]:
# ====================== 1. 导入依赖 ======================
from langchain_core.prompts import PromptTemplate  # 提示词模板
from langchain_community.chat_models import ChatZhipuAI        # 智谱模型
from langchain_core.output_parsers import StrOutputParser  # 输出解析器
from langchain.chains import LLMChain             # LLMChain 核心链
import os
from dotenv import load_dotenv
load_dotenv()


# ====================== 3. 初始化 智谱模型 ======================
llm = ChatZhipuAI(model="glm-4.5-air", api_key=os.getenv("ZHIPUAI_API_KEY"), temperature=0.7)


# ====================== 4. 定义提示模板 ======================
prompt = PromptTemplate(
    input_variables=["question"],  # 输入变量
    template="你是AI助手，请回答问题：{question}"  # 模板
)

# ====================== 5. 初始化输出解析器 ======================
parser = StrOutputParser()

# ====================== 6. 创建 LLMChain（核心） ======================
# LLMChain 继承自 Chain 基类
# 封装了：prompt → llm → parser 完整流程  老式 LLMChain
# LLMChain 从 LangChain 0.1.17 版本开始被标记为 弃用（Deprecated）
# 在未来 1.0 版本中会被彻底移除
# 官方推荐用 RunnableSequence（管道 | 写法） 替代
# chain = LLMChain(
#     prompt=prompt,        # 绑定模板
#     llm=llm,              # 绑定智谱模型
#     output_parser=parser, # 绑定解析器
#     verbose=True          # 打印执行过程（学习用）
# )

# 4. 管道串联（替代旧版 LLMChain）
# 底层就是 RunnableSequence，和 LLMChain 等价
chain = prompt | llm | parser


# ====================== 7. 调用执行 ======================
# invoke 是 Chain 基类提供的统一入口
# 输入：字典
# 输出：字典
result = chain.invoke({
    "question": "什么是 LLMChain？用通俗的话讲"
})

# 输出结果
print("\n最终回答：", result)


最终回答： 通俗来说，**LLMChain 就像给“大语言模型（LLM）”配了一个“智能助手流程包”**。  

你想想，LLM（比如 ChatGPT）本身很厉害，但直接让它干活，有时候会“手忙脚乱”——比如：  
- 你想让它写一封邮件，但它不知道收件人是谁、邮件要说什么；  
- 你想让它根据聊天记录回答问题，但它记不住之前聊了啥；  
- 你想让它先分析数据再写报告，但它不会自己处理数据……  

这时候，**LLMChain 就派上用场了**。它像一个“中间管理者”，把“干活需要的前置步骤”和“LLM本身”串成一条“流水线”，让事情变得更有条理。  


### 举个具体例子：让 LLM 帮你写一封个性化邮件  
单独让 LLM 写邮件，你可能得说：“帮我写一封邮件，收件人是张三，主题是项目进度，内容是告诉他上周完成了第一版，这周要测试，语气正式点”——太啰嗦了，而且容易漏信息。  

如果用 **LLMChain**，流程会变成这样：  
1. **第一步：收集信息**（比如从你之前的对话里提取：“收件人是张三，他是项目负责人，我们上周完成了第一版开发，这周要开始测试，希望他配合资源”）——这部分叫“Prompt 模板”，相当于给 LLM 准备好“原材料清单”；  
2. **第二步：让 LLM 加工**（把清单里的信息塞进提示词，让 LLM 写邮件）——LLM 拿到清单，自动生成一封结构完整、语气合适的邮件；  
3. **第三步：可能再加点“调料”**（比如检查格式、根据你的反馈修改）——如果邮件太长，Chain 可以再让 LLM 摘要；如果语气不对，Chain 可以再调整提示词让 LLM 改。  


### 再举个带“记忆”的例子：聊天机器人  
单独让 LLM 聊天，它每次都像“失忆”，你问“我们上次聊到哪了？”，它可能说“抱歉，我没记住”。  

但如果用 **LLMChain + 记忆模块**，流程就变成了：  
1. **你说话** → Chain 把你的话存进“记忆库”；  
2. **LLM 回复前** → Chain 先从记忆库里翻出之前的对话，和你的新问题一起给 LLM；  
3. **LLM 回复** → Chain 再把回复存进记忆库。  

这样，LLM 就像有了“短期记忆”，能像真人聊天一样，记得上下文。  


### 